In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow_privacy.privacy.optimizers.dp_optimizer_keras import DPKerasAdamOptimizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pathlib import Path

df = pd.read_csv("../data/processed/clean_expenses.csv")
X = df.drop(columns=["monthly_expense"]).values
y = df["monthly_expense"].values

In [ ]:
# Preprocess
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Model
model = keras.Sequential([
    keras.layers.Dense(64, activation="relu", input_shape=(X.shape[1],)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1)
])

In [ ]:
# DP Optimizer
optimizer = DPKerasAdamOptimizer(
    l2_norm_clip=1.0,
    noise_multiplier=1.1,
    num_microbatches=64,
    learning_rate=0.001
)
model.compile(optimizer=optimizer, loss="mse", metrics=["mae"])

In [ ]:
# Train
model.fit(X_train, y_train, epochs=20, batch_size=64, validation_split=0.1)

In [ ]:
# Evaluate
loss, mae = model.evaluate(X_test, y_test)
print(f"DP-SGD Model MAE: {mae:.2f}")
model.save("../experiments/privacy_nn")